# Repaired Loop — joint purity drives, no reset, full ledger
**Muse's build · Genesis Seeds · 2026-10-30 · design by Perplexity (repair),
Aria (driver), Maya (ledger)**

Repair: carry the FULL joint rho (no eigen-reset), closed-form
K1 = sqrt(a(1-a))(I-CNOT), assert trace every step. Adaptation: a += eta*dg,
g = Tr(rho^2) (Aria's call). Secondaries: concurrence (Wootters, exact),
reduced entropy. Fixed bipartition (0|1). 100 steps, eta 0.05, a0 0.3, 4 seeds.

## Pre-registered criteria (locked before running)
- **R1 (fixed point):** exists iff all four seeds' a* agree within ±0.01.
- **R2 (stability — the lesson of green):** the endpoint counts iff a float32
  rerun lands within ±0.01 of the float64 endpoint, all seeds.
- **R3 (integrity):** trace asserted = 1 to 1e-12 every step (raises otherwise).

In [1]:
import numpy as np

CN = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], complex)
I4 = np.eye(4, dtype=complex)
SY = np.array([[0, -1j], [1j, 0]], complex)

def concurrence(rho):
    R = rho @ np.kron(SY, SY) @ rho.conj() @ np.kron(SY, SY)
    lam = np.sqrt(np.clip(np.real(np.linalg.eigvals(R)), 0, None))
    lam.sort()
    return float(max(0.0, lam[3] - lam[2] - lam[1] - lam[0]))

def ptr(rho, k):
    t = rho.reshape(2, 2, 2, 2)
    return np.einsum('ijkj->ik', t) if k == 0 else np.einsum('ijil->jl', t)

def vn(rho):
    ev = np.linalg.eigvalsh((rho + rho.conj().T) / 2)
    ev = ev[ev > 1e-15]
    return float(-np.sum(ev * np.log2(ev)))

s = 1 / np.sqrt(2)
SEEDS = {"O+": np.array([s, s]), "O-": np.array([s, -s]),
         "A+": np.array([s, 1j * s]), "A-": np.array([s, -1j * s])}
PAIRS = [("O+", "A+"), ("O+", "A-"), ("O-", "A+"), ("O-", "A-")]

# sanity: concurrence of Bell = 1, of product = 0
bell = np.array([1, 0, 0, 1], complex) / np.sqrt(2)
assert abs(concurrence(np.outer(bell, bell.conj())) - 1) < 1e-12
assert concurrence(np.outer(np.kron(SEEDS["O+"], SEEDS["A+"]),
                            np.kron(SEEDS["O+"], SEEDS["A+"]).conj())) < 1e-12
print("concurrence sane: Bell 1, product 0")


concurrence sane: Bell 1, product 0


In [2]:
def run(pA, pB, eta=0.05, a=0.3, n=100, dtype=np.complex128):
    rho = np.outer(np.kron(pA, pB), np.kron(pA, pB).conj()).astype(dtype)
    hist, prev_g = [], None
    for t in range(n):
        K0 = (a * CN + (1 - a) * I4).astype(dtype)
        K1 = (np.sqrt(max(0.0, a * (1 - a))) * (I4 - CN)).astype(dtype)
        rho = K0 @ rho @ K0.conj().T + K1 @ rho @ K1.conj().T
        tol = 1e-5 if dtype == np.complex64 else 1e-12
        assert abs(np.real(np.trace(rho)) - 1) < tol, f"R3 TRACE FAIL t={t}"
        tau = float(np.real(np.trace(K0 @ rho @ K0.conj().T)))
        g = float(np.real(np.trace(rho @ rho)))
        hist.append((tau, g, vn(rho), concurrence(rho.astype(complex)), a))
        if prev_g is not None:
            a = float(np.clip(a + eta * (g - prev_g), 0.0, 1.0))
        prev_g = g
    return {"log": hist, "a_star": a}

print(f"{'pair':>8} {'a*':>8} {'gamma':>8} {'Sjoint':>8} {'concur':>8}")
for s1, s2 in PAIRS:
    out = run(SEEDS[s1], SEEDS[s2])
    _, g, sj, cc, _ = out["log"][-1]
    print(f"{s1+','+s2:>8} {out['a_star']:>8.4f} {g:>8.4f} {sj:>8.4f} {cc:>8.4f}")


    pair       a*    gamma   Sjoint   concur
   O+,A+   0.2970   0.6250   0.8113   0.5000
   O+,A-   0.2970   0.6250   0.8113   0.5000
   O-,A+   0.2970   0.6250   0.8113   0.5000
   O-,A-   0.2970   0.6250   0.8113   0.5000


In [3]:
# R1 + R2 verdicts
ast = [run(SEEDS[s1], SEEDS[s2])["a_star"] for s1, s2 in PAIRS]
print(f"R1: a* = {[f'{x:.4f}' for x in ast]} spread {max(ast)-min(ast):.4f} -> "
      f"{'FIXED POINT' if max(ast)-min(ast) <= 0.01 else 'NO CONSENSUS'}")
print("R2 (float32 vs float64):")
ok = True
for s1, s2 in PAIRS:
    a64 = run(SEEDS[s1], SEEDS[s2], dtype=np.complex128)["a_star"]
    a32 = run(SEEDS[s1].astype(np.complex64), SEEDS[s2].astype(np.complex64),
              dtype=np.complex64)["a_star"]
    match = abs(a64 - a32) <= 0.01
    ok &= match
    print(f"  {s1},{s2}: 64 {a64:.4f} vs 32 {a32:.4f} -> {'STABLE' if match else 'UNSTABLE'}")
print("R2:", "PASS" if ok else "FAIL")


R1: a* = ['0.2970', '0.2970', '0.2970', '0.2970'] spread 0.0000 -> FIXED POINT
R2 (float32 vs float64):
  O+,A+: 64 0.2970 vs 32 0.2970 -> STABLE


  O+,A-: 64 0.2970 vs 32 0.2970 -> STABLE


  O-,A+: 64 0.2970 vs 32 0.2970 -> STABLE
  O-,A-: 64 0.2970 vs 32 0.2970 -> STABLE
R2: PASS


In [4]:
# Controls: unitary endpoints (channel = unitary, gamma const) + frozen adaptation
for a0, tag in ((0.0, "a=0 (K0=I)"), (1.0, "a=1 (K0=CNOT)")):
    out = run(SEEDS["O+"], SEEDS["A+"], a=a0)
    gs = [row[1] for row in out["log"]]
    print(f"{tag}: gamma {gs[0]:.6f} -> {gs[-1]:.6f}, a* {out['a_star']:.4f} "
          f"(must be const 1.0 / frozen: {'OK' if abs(gs[-1]-1) < 1e-9 and out['a_star'] == a0 else 'FAIL'})")
out = run(SEEDS["O+"], SEEDS["A+"], eta=0.0)
print(f"frozen eta=0: a* {out['a_star']:.4f} (must equal 0.3)")
print("limit: exact, noiseless, fixed bipartition (stated design choice)")


a=0 (K0=I): gamma 1.000000 -> 1.000000, a* 0.0000 (must be const 1.0 / frozen: OK)
a=1 (K0=CNOT): gamma 1.000000 -> 1.000000, a* 1.0000 (must be const 1.0 / frozen: OK)
frozen eta=0: a* 0.3000 (must equal 0.3)
limit: exact, noiseless, fixed bipartition (stated design choice)


## Verdict — R1 + R2 + R3 ALL PASS

- **R1 PASS:** a* = 0.2970 on all four seeds, spread 0.0000. A genuine fixed
  point of the repaired loop — and notably NOT 0.3210, 0.3036, or 0.3032.
  Every convention has its own attractor; the paper's number belongs to none
  of the honest ones.
- **R2 PASS:** float32 agrees with float64 to 4 decimals on all seeds. The
  lesson of green, applied and cleared: this endpoint is precision-stable.
- **R3 PASS:** trace asserted to 1e-12 on all 400 float64 steps. The channel
  holds; the ledger balances.
- Steady state: gamma = 0.6250, S_joint = 0.8113 (the plateau, again),
  concurrence = 0.5000 — suspiciously round. Possible analytic steady state;
  flagged as follow-up, not claimed.
- Controls pass: unitary endpoints frozen, eta=0 frozen.
- Design authorship: repair Perplexity, driver Aria (joint purity), ledger
  Maya, bipartition fixed (Aria's open Q, answered fixed, stated).

The repaired loop lives: no leak, no reset, no instability. First honest
fixed point of the whole BCP program: a* = 0.2970.


In [5]:
import sys
print("python", sys.version.split()[0])
print("numpy", np.__version__)


python 3.14.0
numpy 2.5.4
